# 01 — Siapkan data MAM, geometri, dan kurva HVSR

**Urutan kerja:** `01 MiniSEED 3 komponen + koordinat → 02 SPAC dan dispersi → 03 inversi Vs → 04 penyesuaian HVSR → 05 pembanding mHVSR`. Jalankan dari folder utama proyek dan ulangi notebook secara berurutan bila input pada tahap sebelumnya berubah. Setiap tahap menyimpan hasil di `outputs/<site_id>/<nomor>/`; notebook berikutnya membacanya otomatis. Berkas rekaman MiniSEED adalah satu-satunya input gelombang yang harus disediakan untuk tahap 01–02. Bobot model lama baru dibaca pada tahap 05.

**Yang Anda isi di sel berikut:** folder berisi 12 MiniSEED (empat sensor × komponen Z, N, E), koordinat X/Y dalam meter untuk Solo1–Solo4, sistem koordinat bila diketahui, elevasi tiap sensor, stasiun untuk HVSR, dan parameter pengolahan HVSR. Nama berkas rekaman harus diawali `Solo1_` sampai `Solo4_`, dengan kanal `GHZ`, `GHN`, dan `GHE`. Nilai bawaan adalah data pilot Solo; ganti koordinat dengan hasil survei lapangan bila tersedia. Berkas konfigurasi site di luar notebook tidak diperlukan. Sel ini menyimpan salinan input sebagai `site_inputs.json` agar notebook 02 memakai nilai yang sama.

**Gagasan metode:** inventaris memeriksa kanal, waktu, sampling, dan mutu amplitudo. HVSR membandingkan spektrum komponen horizontal terhadap vertikal pada satu sensor; kurva ini memberi petunjuk frekuensi resonansi, bukan profil Vs atau Vs30 secara langsung. Kurva dihitung dari jendela 3 komponen dan diberi status QC. Kesamaan waktu pada header belum membuktikan tidak ada drift jam fisik.

**Batas pilot:** koordinat Solo1 yang digeser ke pusat segitiga adalah hipotesis komputasi, bukan posisi pengukuran. Data mentah tidak diubah. Keluaran Geopsy lama tidak dipakai sebagai input.

**Acuan MAM:** [Hayashi et al. (2022)](https://doi.org/10.1007/s10950-021-10051-y), terutama §6–7. Komponen HVSR/QC SESAME tetap memakai acuan khususnya.

Koordinat hasil pemindahan komputasional adalah skenario, bukan posisi sensor saat perekaman. Keselarasan header tidak membuktikan sinkronisasi jam fisik.


## Input pengguna — ubah nilai pada sel berikut sebelum menjalankan notebook

`MINISEED_DIRECTORY` adalah path relatif dari root proyek. `COORDINATES_M` memakai urutan `(X, Y)` dalam meter. `COORDINATE_CRS` boleh `"unverified"` bila belum diketahui. `ARRAY_GEOMETRY` memilih koordinat yang dipakai notebook 02: `"triangle_hypothesis"` memakai Solo1 pada centroid Solo2–Solo4, sedangkan `"supplied"` memakai keempat titik persis seperti dimasukkan. Elevasi tidak dipakai dalam SPAC atau model mHVSR satu masukan; nilainya dicatat untuk konteks lokasi. Pengaturan `HVSR_PROFILE_SETTINGS` menentukan window, penghalusan, penolakan window, dan grid frekuensi.

In [ ]:
SITE_ID = "solo_pilot"
MINISEED_DIRECTORY = "data_test/MAM/mtr_fix"
COORDINATES_M = {
    "Solo1": (570738.0, 9138298.0),
    "Solo2": (570735.0, 9138295.0),
    "Solo3": (570739.0, 9138296.0),
    "Solo4": (570737.0, 9138299.0),
}
COORDINATE_CRS = "verified"
ELEVATION_M = {station: 791.66 for station in COORDINATES_M}
ARRAY_GEOMETRY = "triangle_hypothesis"
HVSR_STATION = "Solo1"

HVSR_PROFILE_SETTINGS = {
    "profile_id": "training_v1",
    "purpose": "Deterministic HVSR curve from the Solo MAM 3C recording",
    "hvsrpy_version": "2.0.0",
    "windowing": {"mode": "fixed_duration", "window_length_s": 60,
                  "detrend": "linear", "taper": "tukey", "taper_width": 0.2,
                  "significant_cycles": 15, "minimum_windows": 15},
    "smoothing": {"operator": "konno_and_ohmachi", "bandwidth": 40,
                  "internal_grid_min_hz": 0.05, "internal_grid_max_hz": 50,
                  "internal_grid_points": 256},
    "rejection": {"method": "frequency_domain", "n_sigma": 2.0,
                  "maximum_iterations": 50, "minimum_accepted_windows": 15,
                  "minimum_accepted_fraction": 0.5},
    "model_grid": {"min_hz": 0.3, "max_hz": 50, "points": 35,
                   "allow_extrapolation": False},
    "method_to_combine_horizontals": "geometric_mean",
    "distribution": "lognormal",
}

In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
from datetime import UTC, datetime
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import obspy
import pandas as pd
import scipy
import hvsrpy
from scipy.signal import resample_poly

from mhvsr_vs30.io.base import ThreeComponentRecord
from mhvsr_vs30.mam.pilot import center_on_outer_centroid, pair_distances
from mhvsr_vs30.preprocessing.config import PreprocessingProfile
from mhvsr_vs30.preprocessing.pipeline import process_recording

ROOT = Path.cwd()
RAW_DIR = ROOT / MINISEED_DIRECTORY
OUT = ROOT / 'outputs' / SITE_ID / '01'
FIG = OUT / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
assert RAW_DIR.is_dir(), f'MiniSEED folder missing: {RAW_DIR}'
assert set(COORDINATES_M) == {'Solo1', 'Solo2', 'Solo3', 'Solo4'}
assert all(len(xy) == 2 and np.isfinite(xy).all() for xy in COORDINATES_M.values())
assert all(float(value) == float(value) for value in ELEVATION_M.values())
assert set(ELEVATION_M) == set(COORDINATES_M)
assert HVSR_STATION in COORDINATES_M
assert ARRAY_GEOMETRY in {'triangle_hypothesis', 'supplied'}
CONFIG = {
    'site_id': SITE_ID,
    'raw_directory': MINISEED_DIRECTORY,
    'measured_coordinates': COORDINATES_M,
    'coordinate_reference_system': COORDINATE_CRS,
    'station_elevation_m': ELEVATION_M,
    'geometry_used_for_spac': ARRAY_GEOMETRY,
    'hvsr_station': HVSR_STATION,
}
(OUT / 'site_inputs.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print('Site:', CONFIG['site_id'], '| output:', OUT)

## 1. Geometri: koordinat yang diberikan dan skenario segitiga

Keempat koordinat asli membentuk enam pasangan. Solo2–Solo4 menjadi tiga titik tepi; Solo1 pada skenario rancangan digeser ke centroid ketiganya. Perubahan itu hanya pada salinan koordinat di memori. Jarak antarsensor dihitung langsung dari koordinat. Grup pasangan pusat–tepi dan tepi–tepi ditandai menurut posisi Solo1 pada skenario segitiga. Unit meter mengikuti informasi pengguna dan konsisten dengan skala gambar; CRS masih belum terverifikasi.


In [ ]:
measured = {name: tuple(map(float, xy)) for name, xy in CONFIG['measured_coordinates'].items()}
modeled = center_on_outer_centroid(measured, 'Solo1', ('Solo2', 'Solo3', 'Solo4'))
scenarios = {'supplied': measured, 'triangle_hypothesis': modeled}

coordinate_rows = []
pair_rows = []
for scenario, coordinates in scenarios.items():
    for station, (x, y) in coordinates.items():
        coordinate_rows.append(dict(scenario=scenario, station=station, x_m=x, y_m=y,
                                    status=('modeled_hypothesis' if scenario == 'triangle_hypothesis' and station == 'Solo1'
                                            else 'user_supplied_unchanged' if scenario == 'triangle_hypothesis'
                                            else 'user_supplied')))
    for left, right, distance in pair_distances(coordinates):
        pair_rows.append(dict(scenario=scenario, station_a=left, station_b=right,
                              distance_m=distance, pair_group=('center_outer' if scenario == 'triangle_hypothesis' and 'Solo1' in (left, right) else 'solo1_other' if 'Solo1' in (left, right) else 'outer_outer')))
coordinates_df = pd.DataFrame(coordinate_rows)
geometry_df = pd.DataFrame(pair_rows)
coordinates_df.to_csv(OUT / 'station_coordinates.csv', index=False)
geometry_df.to_csv(OUT / 'array_geometry.csv', index=False)
print(geometry_df.to_string(index=False))
print('Solo1 modeled displacement (m):', np.hypot(*(np.subtract(modeled['Solo1'], measured['Solo1']))))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, (scenario, coordinates) in zip(axes, scenarios.items()):
    x0 = min(x for x, _ in coordinates.values())
    y0 = min(y for _, y in coordinates.values())
    local = {name: (x - x0, y - y0) for name, (x, y) in coordinates.items()}
    for name, (x, y) in local.items():
        ax.scatter(x, y, c='tab:blue' if name != 'Solo1' else 'tab:red')
        ax.annotate(name, (x, y), xytext=(4, 4), textcoords='offset points')
    ax.plot([local[n][0] for n in ('Solo2', 'Solo3', 'Solo4', 'Solo2')],
            [local[n][1] for n in ('Solo2', 'Solo3', 'Solo4', 'Solo2')], 'k--', alpha=0.6)
    ax.set(title=scenario, xlabel=f'X - {x0:g} (m)', ylabel=f'Y - {y0:g} (m)', aspect='equal')
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG / 'array_geometry_scenarios.png', dpi=160)
plt.show()

## 2. Inventaris MiniSEED dan tumpang tindih waktu

Header dibaca tanpa memuat seluruh waveform. Kesamaan sample rate dan indeks waktu pada grid diuji. Kecocokan header tidak membuktikan akurasi jam fisik atau tidak adanya clock drift.


In [ ]:
header_rows = []
for path in sorted(RAW_DIR.glob('*.mseed')):
    stream = obspy.read(str(path), headonly=True)
    assert len(stream) == 1, f'{path.name}: expected one trace per file'
    tr = stream[0]
    alias = path.name.split('_')[0]
    header_rows.append(dict(station_alias=alias, network=tr.stats.network,
                            station_id=tr.stats.station, channel=tr.stats.channel,
                            component=tr.stats.channel[-1], relative_path=path.relative_to(ROOT).as_posix(),
                            start_utc=str(tr.stats.starttime), end_utc=str(tr.stats.endtime),
                            sampling_rate_hz=float(tr.stats.sampling_rate), n_samples=int(tr.stats.npts),
                            duration_s=float(tr.stats.endtime - tr.stats.starttime),
                            file_bytes=path.stat().st_size,
                            input_sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
inventory = pd.DataFrame(header_rows)
component_sets = inventory.groupby('station_alias')['component'].apply(set)
assert len(inventory) == 12 and all(parts == {'Z', 'N', 'E'} for parts in component_sets)
assert not inventory.duplicated(['station_alias', 'component']).any()
assert inventory.sampling_rate_hz.nunique() == 1
fs = float(inventory.sampling_rate_hz.iloc[0])
inventory['header_sample_count_consistent'] = np.isclose(
    inventory.duration_s * fs + 1, inventory.n_samples, atol=1e-5)
assert inventory.header_sample_count_consistent.all()
common_start = max(obspy.UTCDateTime(t) for t in inventory.start_utc)
common_end = min(obspy.UTCDateTime(t) for t in inventory.end_utc)
assert common_end > common_start
inventory['start_offset_samples_from_common'] = [float((obspy.UTCDateTime(t) - common_start) * fs)
                                                  for t in inventory.start_utc]
inventory['start_on_common_sample_grid'] = np.isclose(inventory.start_offset_samples_from_common,
                                                       np.rint(inventory.start_offset_samples_from_common), atol=1e-6)
inventory.to_csv(OUT / 'station_inventory.csv', index=False)
print(inventory[['station_alias','station_id','channel','start_utc','end_utc','sampling_rate_hz','n_samples']].to_string(index=False))
print('Common UTC interval:', common_start, 'to', common_end,
      '| duration:', round(float(common_end-common_start), 3), 's')
print('All starts on shared sample grid:', bool(inventory.start_on_common_sample_grid.all()))


## 3. Waveform QC seluruh kanal

Pemeriksaan numerik ini mendeteksi nilai non-finite, fraksi sampel nol/konstan, pengulangan nilai ekstrem, dan rasio puncak terhadap persentil 99,9. Ini indikator untuk ditinjau, bukan keputusan membuang kanal. Trace dibaca satu per satu agar penggunaan memori terbatas.


In [ ]:
qc_rows = []
preview = {}
for row in inventory.itertuples(index=False):
    trace = obspy.read(str(ROOT / row.relative_path))[0]
    masked_fraction = float(np.mean(np.ma.getmaskarray(trace.data)))
    values = np.asarray(trace.data, dtype=np.float64)
    finite = np.isfinite(values)
    clean = values[finite]
    assert clean.size > 0, f'{row.relative_path}: no finite samples'
    absvalues = np.abs(clean)
    p999 = float(np.percentile(absvalues, 99.9))
    lo, hi = float(clean.min()), float(clean.max())
    qc_rows.append(dict(station_alias=row.station_alias, channel=row.channel,
                        finite_fraction=float(finite.mean()), masked_fraction=masked_fraction,
                        zero_fraction=float(np.mean(clean == 0)),
                        repeated_extrema_fraction=float(np.mean((clean == lo) | (clean == hi))),
                        flat_step_fraction=float(np.mean(np.diff(clean) == 0)),
                        peak_abs=float(absvalues.max()), p999_abs=p999,
                        peak_to_p999=float(absvalues.max() / p999) if p999 > 0 else np.nan,
                        qc_note='indicators_only; clipping and clock drift require review'))
    if row.component == 'Z':
        step = max(1, len(clean) // 6000)
        preview[row.station_alias] = (np.arange(0, len(clean), step) / fs, clean[::step])
qc = pd.DataFrame(qc_rows)
qc.to_csv(OUT / 'waveform_qc.csv', index=False)
print(qc.to_string(index=False))
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, name in zip(axes, sorted(preview)):
    t, values = preview[name]
    scale = np.percentile(np.abs(values), 99) or 1.0
    ax.plot(t, values / scale, lw=0.4)
    ax.set_ylabel(name + ' Z / p99')
    ax.grid(alpha=0.2)
axes[-1].set_xlabel('Seconds since channel start')
fig.tight_layout()
fig.savefig(FIG / 'waveform_qc.png', dpi=150)
plt.show()


## 4. HVSR awal dari Solo1, satu sensor di dalam array

Ini HVSR **Solo1 pada tanggal MAM**,. Tiga komponen diturunkan dari 2.000 Hz menjadi 200 Hz dengan `scipy.signal.resample_poly` (filter anti-alias bawaan), lalu diolah menggunakan `hvsrpy` melalui pipeline repositori dan parameter HVSR pada sel input di atas. Pemilihan Solo1 tidak membuktikan ia berada di pusat array fisik. Notebook contoh penghapusan respons menunjukkan rancangan keluaran m/s, tetapi nama sensor dan tanggal contohnya berbeda dari berkas pilot; satuan berkas pilot belum terverifikasi.

In [ ]:
station = CONFIG['hvsr_station']
component_data = {}
for component in ('Z', 'N', 'E'):
    row = inventory[(inventory.station_alias == station) & (inventory.component == component)].iloc[0]
    trace = obspy.read(str(ROOT / row.relative_path))[0]
    component_data[component] = resample_poly(np.asarray(trace.data, dtype=np.float64), 1, 10)
assert len({len(x) for x in component_data.values()}) == 1
profile = PreprocessingProfile(**HVSR_PROFILE_SETTINGS)
profile = profile.model_copy(update={'config_hash': profile.identity()})
(OUT / 'hvsr_profile_inputs.json').write_text(
    json.dumps(HVSR_PROFILE_SETTINGS, indent=2), encoding='utf-8')
start = obspy.UTCDateTime(inventory[inventory.station_alias == station].start_utc.iloc[0]).datetime
record = ThreeComponentRecord(z=component_data['Z'], north=component_data['N'],
                              east=component_data['E'], sampling_rate_hz=fs/10,
                              start_time=start.replace(tzinfo=start.tzinfo or UTC),
                              units='unknown_common_source_unit',
                              source_channels={'Z':'GHZ','N':'GHN','E':'GHE'},
                              recording_id='solo_pilot_mam_Solo1_20260812')
processed = process_recording(record, profile)
hvsr_df = pd.DataFrame(dict(frequency_hz=processed.frequency_hz,
                            hvsr_mean=processed.mean_curve,
                            hvsr_log_std=processed.log_std_curve,
                            station_alias=station, input_level='derived_from_3c_mam',
                            preprocessing_profile=profile.profile_id))
hvsr_df.to_csv(OUT / 'hvsr_observed.csv', index=False)
print('HVSR windows:', len(processed.accepted_window_mask),
      'accepted:', int(np.count_nonzero(processed.accepted_window_mask)),
      '| flags:', processed.qc_flags)
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(hvsr_df.frequency_hz, hvsr_df.hvsr_mean, label='Solo1 HVSR mean')
ax.set(xlabel='Frequency (Hz)', ylabel='H/V', title='Observed HVSR from Solo1 MAM recording')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIG / 'hvsr_qc.png', dpi=160)
plt.show()

## 5. Ringkasan dan gerbang Notebook 01

Artefak keluaran berisi koordinat asli dan hipotesis secara terpisah, inventaris dan QC waveform, serta HVSR Solo1. **Status tahap ini:** inspeksi dan HVSR awal dapat direproduksi; posisi sensor yang dikoreksi belum terverifikasi sebagai posisi lapangan. Kurva dispersi dan model Vs belum dihitung. Review geometri dan keterkaitan lokasi HVSR diperlukan sebelum Notebook 02.

In [ ]:
metadata = {
    'method_revision': 'hayashi_2022_v1', 'run_state': 'complete',
    'mam_reference_doi': '10.1007/s10950-021-10051-y',
    'site_id': CONFIG['site_id'], 'input_level': 'raw_mam_3c',
    'processed_at_utc': datetime.now(UTC).isoformat(),
    'source_directory': RAW_DIR.relative_to(ROOT).as_posix(),
    'source_files': inventory[['relative_path', 'input_sha256']].to_dict('records'),
    'python_version': platform.python_version(), 'obspy_version': obspy.__version__,
    'scipy_version': scipy.__version__, 'numpy_version': np.__version__,
    'hvsrpy_version': hvsrpy.__version__, 'sampling_rate_hz': fs,
    'common_start_utc': str(common_start), 'common_end_utc': str(common_end),
    'all_starts_on_common_sample_grid': bool(inventory.start_on_common_sample_grid.all()),
    'all_header_sample_counts_consistent': bool(inventory.header_sample_count_consistent.all()),
    'physical_clock_drift_checked': False,
    'coordinate_reference_system': CONFIG['coordinate_reference_system'],
    'geometry_scenarios': {'supplied': 'user_supplied', 'triangle_hypothesis': 'modeled_not_measured'},
    'solo1_modeled_displacement_m': float(np.hypot(*(np.subtract(modeled['Solo1'], measured['Solo1'])))),
    'external_processed_reference_required': False,
    'hvsr_station': station, 'hvsr_input_level': 'derived_from_3c_mam',
    'hvsr_profile_id': profile.profile_id, 'hvsr_profile_hash': profile.config_hash,
    'hvsr_resample_original_hz': fs, 'hvsr_resample_final_hz': fs/10,
    'hvsr_accepted_windows': int(np.count_nonzero(processed.accepted_window_mask)),
    'hvsr_qc_flags': list(processed.qc_flags),
    'spac_ready': False,
    'blocking_review_items': ['confirm true sensor coordinates', 'confirm CRS and station mapping',
                              'review waveform QC and physical clock drift', 'assess HVSR spatial representativeness'],
}
(OUT / 'preprocessing_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Notebook 01 outputs saved to:', OUT)
print('SPAC ready:', metadata['spac_ready'], '| review:', metadata['blocking_review_items'])
